# Analisis por familia de difusion - GenImage (E2)

Notebook ligero: lee las predicciones y metricas ya guardadas en Drive por `notebook_genimage.ipynb` y produce el material analitico que faltaba (tablas por familia, figuras complementarias y esqueleto del informe).

**No descarga el dataset ni usa GPU**. Solo NumPy/Pandas/Sklearn/Matplotlib. Tiempo total ~2-3 min.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, re, json, glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, confusion_matrix

plt.rcParams.update({'font.family': 'serif', 'axes.spines.top': False, 'axes.spines.right': False})

CKPT_BASE = '/content/drive/MyDrive/tfm-checkpoints'
GENIMAGE_DIR = Path(CKPT_BASE) / 'figuras_memoria_genimage'
OUT_DIR = GENIMAGE_DIR
print(f'Lectura/escritura: {OUT_DIR}')

## 2. Carga de ficheros desde Drive

Leer:
- `predicciones_{resnet50,vit,ufd}.pt` (3 ficheros con `y_true`, `y_score`, `generators` por muestra).
- `metricas_genimage.json` (agregado + por arq, ya calculadas).
- `{modelo}_*/metrics.json` (E1 + E1b por modelo, para construir la tabla del degradado E1->E1b->E2).

In [ ]:
# Predicciones por muestra
MODELOS = ['ResNet-50', 'ViT-B/16', 'UniversalFakeDetect']
KEY_MAP = {'ResNet-50': 'resnet50', 'ViT-B/16': 'vit', 'UniversalFakeDetect': 'ufd'}
PREFIX_MAP = {'ResNet-50': 'resnet50', 'ViT-B/16': 'vit', 'UniversalFakeDetect': 'universalfakedetect'}
COLORES = {'ResNet-50': '#1976D2', 'ViT-B/16': '#388E3C', 'UniversalFakeDetect': '#F57C00'}

predicciones = {}
for n in MODELOS:
    key = KEY_MAP[n]
    path = OUT_DIR / f'predicciones_{key}.pt'
    d = torch.load(str(path), map_location='cpu')
    predicciones[n] = {
        'y_true':     d['y_true'].numpy() if hasattr(d['y_true'], 'numpy') else np.asarray(d['y_true']),
        'y_score':    d['y_score'].numpy() if hasattr(d['y_score'], 'numpy') else np.asarray(d['y_score']),
        'generators': list(d['generators']),
    }
    print(f'  {n:<22s}  N={len(predicciones[n]["y_true"]):>7d}  ({path.name})')

# Metricas E2 (agregado + por arq) ya calculadas
with (OUT_DIR / 'metricas_genimage.json').open() as f:
    metricas_e2 = json.load(f)

# Localizar metrics.json de E1/E1b por modelo via glob (mismo patron que el notebook principal)
metrics_e1_e1b = {}
for n in MODELOS:
    cands = sorted(glob.glob(f'{CKPT_BASE}/{PREFIX_MAP[n]}_*/metrics.json'))
    if not cands:
        raise FileNotFoundError(f'No hay metrics.json para {n}')
    with open(cands[-1]) as f:
        metrics_e1_e1b[n] = json.load(f)
    print(f'  {n:<22s}  {cands[-1]}')

## 3. Familias de difusion y normalizacion de nombres

Dos cortes complementarios sobre las 6 arquitecturas:

| Corte | Familias |
|---|---|
| **Binario** | LDM (latent diffusion + CLIP) vs no-LDM |
| **Por mecanismo** | LDM / Pixel class-cond / Pixel text-cond / Discrete VQ |

LDM agrupa SD v1.4, SD v1.5 y Wukong porque comparten arquitectura: U-Net sobre latente de VAE + cross-attention con CLIP text encoder (Wukong es fork chino de SD entrenado en LAION-Chinese).

In [ ]:
def normalizar(s):
    """Normaliza un nombre de arquitectura: minusculas, alfanumerico+guion bajo."""
    return re.sub(r'[^a-z0-9]+', '_', s.lower()).strip('_')

# Los nombres reales en GenImage siguen el patron 'imagenet_ai_XXXX_<arq>' o 'imagenet_<arq>',
# y SD viene como 'sdv4'/'sdv5' (no 'v1.4'/'v1.5'). Matcheamos por substring para robustez.

def _familia_binaria(norm):
    if 'sdv' in norm or 'wukong' in norm: return 'LDM'
    if 'adm' in norm or 'glide' in norm or 'vqdm' in norm: return 'no-LDM'
    return None

def _familia_mecanismo(norm):
    if 'sdv' in norm or 'wukong' in norm: return 'LDM'
    if 'adm' in norm:   return 'Pixel class-cond'
    if 'glide' in norm: return 'Pixel text-cond'
    if 'vqdm' in norm:  return 'Discrete VQ'
    return None

def _nombre_bonito(norm):
    if 'sdv4' in norm:   return 'SD v1.4'
    if 'sdv5' in norm:   return 'SD v1.5'
    if 'wukong' in norm: return 'Wukong'
    if 'glide' in norm:  return 'GLIDE'
    if 'vqdm' in norm:   return 'VQDM'
    if 'adm' in norm:    return 'ADM'
    return norm

todas_arqs = sorted(set(predicciones[MODELOS[0]]['generators']))
FAMILIA_BINARIA   = {normalizar(a): _familia_binaria(normalizar(a))   for a in todas_arqs}
FAMILIA_MECANISMO = {normalizar(a): _familia_mecanismo(normalizar(a)) for a in todas_arqs}
NOMBRE_BONITO     = {normalizar(a): _nombre_bonito(normalizar(a))     for a in todas_arqs}

print(f'Arquitecturas encontradas en las predicciones ({len(todas_arqs)}):')
for a in todas_arqs:
    n = normalizar(a)
    print(f'  raw={a!r:<40s}  bonito={NOMBRE_BONITO[n]:<10s}  bin={FAMILIA_BINARIA[n] or "?":<8s}  mec={FAMILIA_MECANISMO[n] or "?"}')

El mapping deberia mostrar las 6 arquitecturas con familia binaria y mecanismo asignadas (sin `?`). Si alguna sale como `?`, anadir el substring correspondiente al matching de las funciones `_familia_*` y re-ejecutar.

In [ ]:
def metricas_subset(y_true, y_score):
    """Calcula AUC, AP, Acc para un subconjunto. Devuelve None si solo hay una clase."""
    if len(set(y_true)) < 2:
        return None
    y_pred = (y_score >= 0.5).astype(int)
    return {
        'auc_roc':           float(roc_auc_score(y_true, y_score)),
        'average_precision': float(average_precision_score(y_true, y_score)),
        'accuracy':          float(accuracy_score(y_true, y_pred)),
        'n_samples':         int(len(y_true)),
        'n_real':            int((y_true == 0).sum()),
        'n_fake':            int((y_true == 1).sum()),
    }

def agregar_por_clave(modelo, clave_dict):
    """Agrupa muestras del modelo por familia (clave_dict[norm_gen]) y calcula metricas."""
    preds = predicciones[modelo]
    y_true  = preds['y_true']
    y_score = preds['y_score']
    gens    = preds['generators']

    grupos = defaultdict(lambda: {'yt': [], 'ys': []})
    for yt, ys, g in zip(y_true, y_score, gens):
        fam = clave_dict.get(normalizar(g))
        if fam is None:
            continue
        grupos[fam]['yt'].append(yt)
        grupos[fam]['ys'].append(ys)

    return {fam: metricas_subset(np.array(d['yt']), np.array(d['ys'])) for fam, d in grupos.items()}

## 4. Tabla maestra por arquitectura

Es el equivalente al `desglose_genimage_por_arquitectura.csv` pero en DataFrame y con la columna `Ganador`.

In [ ]:
filas = []
for arq_raw in todas_arqs:
    norm = normalizar(arq_raw)
    bonito = NOMBRE_BONITO.get(norm, arq_raw)
    fila = {'arq': bonito, 'familia_bin': FAMILIA_BINARIA.get(norm, '?'), 'familia_mec': FAMILIA_MECANISMO.get(norm, '?')}

    aucs = {}
    for n in MODELOS:
        m = metricas_e2['por_arquitectura'].get(n, {}).get(arq_raw)
        if m is None or 'auc_roc' not in m:
            # Fallback: recomputar desde el .pt
            mask = np.array([normalizar(g) == norm for g in predicciones[n]['generators']])
            yt = predicciones[n]['y_true'][mask]
            ys = predicciones[n]['y_score'][mask]
            m = metricas_subset(yt, ys) or {}
        fila[f'{KEY_MAP[n]}_auc'] = m.get('auc_roc', np.nan)
        fila[f'{KEY_MAP[n]}_ap']  = m.get('average_precision', np.nan)
        fila[f'{KEY_MAP[n]}_acc'] = m.get('accuracy', np.nan)
        fila['n_total'] = m.get('n_samples', 0)
        fila['n_fake']  = m.get('n_fake', 0)
        aucs[n] = m.get('auc_roc', np.nan)

    ganador = max(aucs, key=lambda k: aucs[k] if not np.isnan(aucs[k]) else -1)
    fila['ganador'] = ganador
    filas.append(fila)

df_arq = pd.DataFrame(filas).sort_values('resnet50_auc', ascending=False).reset_index(drop=True)
df_arq[['arq', 'familia_bin', 'familia_mec', 'n_total', 'n_fake', 'resnet50_auc', 'vit_auc', 'ufd_auc', 'ganador']].round(4)

## 5. Tabla por familia (binaria: LDM vs no-LDM)

In [ ]:
filas = []
for n in MODELOS:
    por_fam = agregar_por_clave(n, FAMILIA_BINARIA)
    for fam, m in por_fam.items():
        if m is None:
            continue
        filas.append({
            'familia':  fam,
            'modelo':   n,
            'auc_roc':  m['auc_roc'],
            'ap':       m['average_precision'],
            'accuracy': m['accuracy'],
            'n':        m['n_samples'],
        })

df_fam_bin = pd.DataFrame(filas)
pivot_bin = df_fam_bin.pivot(index='familia', columns='modelo', values='auc_roc').reindex(['LDM', 'no-LDM'])[MODELOS].round(4)
print('AUC-ROC por familia binaria:')
print(pivot_bin.to_string())
print()
print('Acc por familia binaria:')
print(df_fam_bin.pivot(index='familia', columns='modelo', values='accuracy').reindex(['LDM', 'no-LDM'])[MODELOS].round(4).to_string())

## 6. Tabla por familia (mecanismo)

In [ ]:
filas = []
for n in MODELOS:
    por_fam = agregar_por_clave(n, FAMILIA_MECANISMO)
    for fam, m in por_fam.items():
        if m is None:
            continue
        filas.append({
            'familia':  fam,
            'modelo':   n,
            'auc_roc':  m['auc_roc'],
            'ap':       m['average_precision'],
            'accuracy': m['accuracy'],
            'n':        m['n_samples'],
        })

df_fam_mec = pd.DataFrame(filas)
orden_mec = ['LDM', 'Pixel class-cond', 'Pixel text-cond', 'Discrete VQ']
pivot_mec = df_fam_mec.pivot(index='familia', columns='modelo', values='auc_roc').reindex(orden_mec)[MODELOS].round(4)
print('AUC-ROC por familia (mecanismo):')
print(pivot_mec.to_string())

## 7. Tabla extendida E1 -> E1b -> E2 (degradado por dominio)

In [ ]:
filas = []
for n in MODELOS:
    e1   = metrics_e1_e1b[n]['e1']
    e1b  = metrics_e1_e1b[n]['e1b']
    e2   = metricas_e2['agregado'][n]
    filas.append({
        'modelo':       n,
        'e1_auc':       e1['auc_roc'],
        'e1b_auc':      e1b['auc_roc'],
        'e2_auc':       e2['auc_roc'],
        'delta_e1_e1b': e1b['auc_roc'] - e1['auc_roc'],
        'delta_e1b_e2': e2['auc_roc'] - e1b['auc_roc'],
        'delta_e1_e2':  e2['auc_roc'] - e1['auc_roc'],
        'e1_acc':       e1['accuracy'],
        'e1b_acc':      e1b['accuracy'],
        'e2_acc':       e2['accuracy'],
    })

df_deg = pd.DataFrame(filas)
print('Degradado por dominio (AUC):')
print(df_deg[['modelo', 'e1_auc', 'e1b_auc', 'e2_auc', 'delta_e1_e1b', 'delta_e1b_e2', 'delta_e1_e2']].round(4).to_string(index=False))
print()
print('Degradado por dominio (Acc):')
print(df_deg[['modelo', 'e1_acc', 'e1b_acc', 'e2_acc']].round(4).to_string(index=False))

## 8. Figuras nuevas

Las 5 figuras del notebook principal cubren agregado + por arq. Estas 3 cubren el angulo de **familias** y se guardan en el mismo directorio.

In [ ]:
# Figura A - Heatmap AUC por familia (mecanismo) x modelo
heat = pivot_mec.values
fig, ax = plt.subplots(figsize=(8, max(3, len(orden_mec) * 0.9)))
sns.heatmap(heat, annot=True, fmt='.4f', cmap='RdYlGn', vmin=0.5, vmax=1.0,
            xticklabels=MODELOS, yticklabels=orden_mec,
            ax=ax, linewidths=0.5, linecolor='white',
            cbar_kws={'label': 'AUC-ROC', 'shrink': 0.7},
            annot_kws={'fontsize': 11, 'fontweight': 'bold'})
ax.set_title('AUC-ROC por familia de difusion (mecanismo) y modelo', fontsize=13, pad=12)
ax.set_xlabel('Modelo detector', fontsize=12)
ax.set_ylabel('Familia generativa', fontsize=12)
ax.tick_params(axis='x', rotation=15, labelsize=10)
ax.tick_params(axis='y', rotation=0, labelsize=10)
plt.tight_layout()
out_path = OUT_DIR / 'heatmap_auc_por_familia_mecanismo.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')

In [ ]:
# Figura B - Barras AUC LDM vs no-LDM por modelo
fig, ax = plt.subplots(figsize=(9, 5.5))
width = 0.35
x = np.arange(len(MODELOS))
ldm    = [pivot_bin.loc['LDM', n]    for n in MODELOS]
noldm  = [pivot_bin.loc['no-LDM', n] for n in MODELOS]

bars1 = ax.bar(x - width/2, ldm,   width, label='LDM (SD v1.4, SD v1.5, Wukong)', color='#5E35B1', alpha=0.9, edgecolor='white')
bars2 = ax.bar(x + width/2, noldm, width, label='no-LDM (ADM, GLIDE, VQDM)',     color='#FB8C00', alpha=0.9, edgecolor='white')

for bars in (bars1, bars2):
    for b in bars:
        ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.005,
                f'{b.get_height():.3f}', ha='center', va='bottom', fontsize=10)

ax.set_xticks(x)
ax.set_xticklabels(MODELOS, fontsize=11)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('AUC-ROC por familia binaria (LDM vs no-LDM)', fontsize=13)
ax.legend(fontsize=10, loc='lower right')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
out_path = OUT_DIR / 'barras_auc_ldm_vs_noldm.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')

In [ ]:
# Figura C - Degradado E1 -> E1b -> E2 por modelo (lineas)
rondas = ['E1\n(ProGAN)', 'E1b\n(GAN cross-arch)', 'E2\n(difusion)']
fig, ax = plt.subplots(figsize=(9, 6))
for n in MODELOS:
    fila = df_deg[df_deg['modelo'] == n].iloc[0]
    valores = [fila['e1_auc'], fila['e1b_auc'], fila['e2_auc']]
    ax.plot(rondas, valores, marker='o', markersize=10, lw=2.5, color=COLORES[n], label=n)
    for x_pos, v in zip(rondas, valores):
        ax.text(x_pos, v + 0.01, f'{v:.3f}', ha='center', va='bottom', fontsize=10, color=COLORES[n])

ax.set_ylim(0.4, 1.05)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('Degradado de AUC por dominio: ProGAN -> GAN cross-arch -> Difusion', fontsize=13)
ax.legend(fontsize=11, loc='lower left')
ax.yaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
out_path = OUT_DIR / 'lineas_degradado_e1_e1b_e2.png'
plt.savefig(str(out_path), dpi=300, bbox_inches='tight')
plt.show()
print(f'Guardado: {out_path}')

## 9. Generacion del esqueleto del informe

Guarda `informe_analisis_resultados_e2.md` en Drive con todas las **tablas rellenas con numeros reales** y las secciones interpretativas marcadas como `[INTERPRETACION PENDIENTE]` para terminarlas en chat.

In [ ]:
def md_table_arq(df):
    rows = ['| Arq | Familia | n_total | n_fake | ResNet AUC | ViT AUC | UFD AUC | Ganador |',
            '|---|---|---:|---:|---:|---:|---:|---|']
    for _, r in df.iterrows():
        rows.append(f"| {r['arq']} | {r['familia_bin']} ({r['familia_mec']}) | {int(r['n_total'])} | {int(r['n_fake'])} | {r['resnet50_auc']:.4f} | {r['vit_auc']:.4f} | {r['ufd_auc']:.4f} | {r['ganador']} |")
    return '\n'.join(rows)

def md_table_pivot(piv, titulo_col='Familia'):
    rows = [f'| {titulo_col} | ' + ' | '.join(MODELOS) + ' |',
            '|---|' + '---:|' * len(MODELOS)]
    for fam, fila in piv.iterrows():
        vals = ' | '.join(f'{fila[n]:.4f}' if not np.isnan(fila[n]) else 'n/a' for n in MODELOS)
        rows.append(f'| {fam} | {vals} |')
    return '\n'.join(rows)

def md_table_degradado(df):
    rows = ['| Modelo | E1 AUC | E1b AUC | E2 AUC | Delta E1->E1b | Delta E1b->E2 | Delta E1->E2 |',
            '|---|---:|---:|---:|---:|---:|---:|']
    for _, r in df.iterrows():
        rows.append(f"| {r['modelo']} | {r['e1_auc']:.4f} | {r['e1b_auc']:.4f} | {r['e2_auc']:.4f} | {r['delta_e1_e1b']:+.4f} | {r['delta_e1b_e2']:+.4f} | {r['delta_e1_e2']:+.4f} |")
    return '\n'.join(rows)

# Numeros para el resumen ejecutivo
agregado = metricas_e2['agregado']
n_total_e2 = agregado[MODELOS[0]]['n_samples']

informe_md = f"""# Informe de analisis de resultados - Cross-generator (E2 / GenImage)

**TFM** - *Deteccion de imagenes sinteticas generadas por modelos de difusion: evaluacion comparativa de arquitecturas de Deep Learning.*

**Alcance**: analisis interpretativo de la evaluacion E2 sobre las 6 arquitecturas de difusion de GenImage. Material de apoyo para la seccion de Resultados (extension cross-generator) de la memoria. Complementa al `informe_analisis_resultados.md` (E1 + E1b).

---

## 0. Aviso preliminar sobre el alcance del testset E2

E2 evalua exclusivamente **modelos de difusion** (no GANs). Las 8 arquitecturas originales de GenImage se reducen a 6 por dos exclusiones deliberadas:

- **BigGAN**: ya cubierta en E1b (CNN_synth_testset) como GAN cross-arch. Su inclusion aqui solo duplicaria la senal de E1b.
- **Midjourney**: excluida por (a) restriccion de disco en Colab Free (>100 GB del zip multi-volumen) y (b) argumento metodologico - su resolucion nativa de 1024 px se destruye al aplicar el resize a 224 px del pipeline unificado, lo que invalidaria la lectura de cualquier resultado. Justificacion detallada en `expEjecucionGenImage.md` sec. 8.1.

Distribucion del testset E2 por arquitectura:

{md_table_arq(df_arq)}

**Total**: {n_total_e2:,} imagenes.

---

## 1. Resumen ejecutivo

### 1.1 Metricas agregadas E2 (todo GenImage val)

| Modelo | AUC | AP | Accuracy | N |
|---|---:|---:|---:|---:|
"""
for n in MODELOS:
    m = agregado[n]
    informe_md += f"| {n} | {m['auc_roc']:.4f} | {m['average_precision']:.4f} | {m['accuracy']*100:.2f} % | {m['n_samples']:,} |\n"

informe_md += f"""
### 1.2 Sintesis

[INTERPRETACION PENDIENTE - escribir en chat con base en los numeros de la tabla 1.1 y la 5.1]

---

## 2. Evaluacion E2 - analisis agregado

### 2.1 Matrices de confusion E2

| Modelo | TN | FP | FN | TP | TPR | FPR |
|---|---:|---:|---:|---:|---:|---:|
"""
for n in MODELOS:
    cm = np.array(agregado[n]['confusion_matrix'])
    tn, fp = int(cm[0, 0]), int(cm[0, 1])
    fn, tp = int(cm[1, 0]), int(cm[1, 1])
    tpr = tp / (tp + fn) if (tp + fn) > 0 else float('nan')
    fpr = fp / (fp + tn) if (fp + tn) > 0 else float('nan')
    informe_md += f"| {n} | {tn:,} | {fp:,} | {fn:,} | {tp:,} | {tpr*100:.2f} % | {fpr*100:.2f} % |\n"

informe_md += f"""
### 2.2 Divergencia AUC vs accuracy

| Modelo | AUC | Accuracy | Brecha AUC -> Acc |
|---|---:|---:|---:|
"""
for n in MODELOS:
    m = agregado[n]
    informe_md += f"| {n} | {m['auc_roc']:.4f} | {m['accuracy']:.4f} | {m['accuracy'] - m['auc_roc']:+.4f} |\n"

informe_md += """
[INTERPRETACION PENDIENTE - comentar (a) ranking, (b) sesgo conservador (ratio FN/FP), (c) calibracion del umbral en cross-paradigma vs cross-arch]

### 2.3 Curvas ROC

Figura ya disponible en `roc_curves_e2_superpuestas.png` (de la primera ejecucion del notebook GenImage).

[INTERPRETACION PENDIENTE - comentar zonas de bajo FPR, cruces entre curvas]

---

## 3. Desglose por arquitectura

### 3.1 Tabla maestra E2

"""
informe_md += md_table_arq(df_arq) + "\n\n"

victorias = df_arq['ganador'].value_counts().to_dict()
informe_md += '**Conteo de victorias por modelo (AUC):** '
informe_md += ', '.join(f'{n} ({victorias.get(n, 0)})' for n in MODELOS) + '.\n\n'

informe_md += '[INTERPRETACION PENDIENTE - patron de victorias, arq mas dificil/facil, inversiones de ranking]\n\n---\n\n'

informe_md += '## 4. Analisis por familia de difusion\n\n'
informe_md += '### 4.1 Binaria - LDM vs no-LDM\n\n'
informe_md += md_table_pivot(pivot_bin, 'Familia') + '\n\n'
informe_md += 'Composicion:\n- **LDM**: SD v1.4, SD v1.5, Wukong (latent diffusion + CLIP text encoder)\n- **no-LDM**: ADM (pixel class-cond), GLIDE (pixel text-cond), VQDM (discrete VQ)\n\n'
informe_md += '[INTERPRETACION PENDIENTE - ¿generalizan mejor dentro de LDM? ¿por que?]\n\n'

informe_md += '### 4.2 Por mecanismo\n\n'
informe_md += md_table_pivot(pivot_mec, 'Familia') + '\n\n'
informe_md += '[INTERPRETACION PENDIENTE - granularidad fina; mencionar que 3 de 4 familias tienen N=1 arq y eso limita generalizacion]\n\n---\n\n'

informe_md += '## 5. Brecha E1 -> E1b -> E2 (degradado por dominio)\n\n'
informe_md += '### 5.1 AUC\n\n'
informe_md += md_table_degradado(df_deg) + '\n\n'

informe_md += '### 5.2 Accuracy\n\n'
informe_md += '| Modelo | E1 Acc | E1b Acc | E2 Acc |\n|---|---:|---:|---:|\n'
for _, r in df_deg.iterrows():
    informe_md += f"| {r['modelo']} | {r['e1_acc']*100:.2f} % | {r['e1b_acc']*100:.2f} % | {r['e2_acc']*100:.2f} % |\n"

informe_md += """
[INTERPRETACION PENDIENTE - este es el highlight: ¿se cumple el ranking de robustez de E1->E1b cuando saltamos a E2? ¿que modelo conserva mejor capacidad al saltar de paradigma generativo?]

---

## 6. Contraste con la hipotesis original

El informe E1/E1b (sec. 10.2) marcaba como **limitacion principal** la ausencia de evaluacion contra modelos de difusion: "la hipotesis teorica original de UFD se prueba mas fuerte si el modelo generaliza bien contra Stable Diffusion / Midjourney / DALL-E". E2 cierra parcialmente ese gap.

[INTERPRETACION PENDIENTE - ¿la hipotesis de Ojha et al. sobre 'universalidad' de CLIP se sostiene al saltar de GAN a difusion? ¿el ranking E1b sigue valido en E2?]

---

## 7. Limitaciones especificas de E2

1. **Midjourney excluida** (resolucion 1024 px destruida por el resize). El paradigma 'difusion comercial state-of-the-art' queda sin evaluar.
2. **BigGAN excluida** del corte E2 (ya estaba en E1b).
3. **Solo split `val/`**, no `train/`. Coherente con el resto del protocolo pero limita N.
4. **Resize agresivo (224 px)** estandar del pipeline; pierde artefactos de alta frecuencia que algunas arqs (especialmente SD a 512 px) podrian exhibir mas claramente a resolucion nativa.
5. **3 de las 4 familias por mecanismo tienen N=1 arquitectura**. La generalizacion de la conclusion por mecanismo es limitada; la binaria (LDM vs no-LDM) es 3 vs 3.
6. **Limitaciones heredadas de E1/E1b**: seed unica, sin bandas de confianza, posible solapamiento dominio CLIP-testset (UFD).

---

## 8. Conclusiones del analisis E2

[PENDIENTE - escribir despues de cerrar las interpretaciones de las secciones 1, 2, 3, 4, 5 y 6]

---

## Apendice - Ficheros de soporte en Drive

Todos en `MyDrive/tfm-checkpoints/figuras_memoria_genimage/`:

- `metricas_genimage.json` - metricas agregadas y por arq.
- `desglose_genimage_por_arquitectura.csv` - tabla maestra plana.
- `predicciones_{resnet50,vit,ufd}.pt` - scores por muestra (para recomputos finos).
- `matrices_confusion_e2.png` - matrices E2.
- `barras_metricas_e2.png` - barras AUC/AP/Acc agregadas.
- `roc_curves_e2_superpuestas.png` - ROC E2.
- `heatmap_auc_por_generador_e2.png` - AUC por arq x modelo.
- `comparativa_gan_vs_difusion.png` - degradado E1/E1b/E2 (barras).
- `heatmap_auc_por_familia_mecanismo.png` - AUC por familia mecanismo x modelo *(nueva)*.
- `barras_auc_ldm_vs_noldm.png` - LDM vs no-LDM *(nueva)*.
- `lineas_degradado_e1_e1b_e2.png` - degradado en lineas *(nueva)*.
"""

md_path = OUT_DIR / 'informe_analisis_resultados_e2.md'
md_path.write_text(informe_md, encoding='utf-8')
print(f'Informe esqueleto guardado en: {md_path}')
print(f'Tamano: {md_path.stat().st_size / 1024:.1f} KB')

## 10. Resumen de outputs generados

In [ ]:
print('Ficheros en', OUT_DIR, ':')
for f in sorted(OUT_DIR.iterdir()):
    size_kb = f.stat().st_size / 1024
    marker = '  (nuevo)' if f.name in ('informe_analisis_resultados_e2.md',
                                        'heatmap_auc_por_familia_mecanismo.png',
                                        'barras_auc_ldm_vs_noldm.png',
                                        'lineas_degradado_e1_e1b_e2.png') else ''
    print(f'  {f.name:<50s}  {size_kb:>8.1f} KB{marker}')